In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline  # Use imblearn's Pipeline
import matplotlib.pyplot as plt
import warnings
from alive_progress import alive_bar
from contextlib import contextmanager
from joblib.parallel import BatchCompletionCallBack
import threading

# Import oversampling techniques from imbalanced-learn
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Load the dataset
# Replace the file path with your actual path
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define the feature indexes to be used as predictors
# Note: Pandas uses 0-based indexing
feature_indexes = [28, 234, 36, 215, 189, 37, 77, 26, 250, 188, 236, 181, 70, 55, 48, 67, 202, 244, 78, 65, 52, 237, 211, 57, 185, 229, 231, 219, 62, 220, 51, 200, 30, 242, 233, 198, 34, 25, 212, 205, 252, 32, 29, 248, 253, 75, 66, 247, 56, 58, 12, 221, 41, 197, 7, 63, 217, 1, 251, 199, 114, 49, 24, 50, 101, 80, 186, 232, 207, 130, 79, 115, 43, 243, 190, 137, 136, 108, 46, 214, 17, 10, 110, 88, 125, 182, 33, 203, 225, 68, 213, 227, 126, 93, 81, 14, 13, 104, 92, 42, 201, 129, 177, 122, 134, 133, 60, 19, 31, 9, 135, 98, 8, 45, 3, 47, 4, 6, 226, 116, 106, 90, 15, 105, 138, 18, 89, 84, 100, 44, 228, 131, 38, 112, 103, 96, 16, 127, 206, 117, 11, 102, 176, 128, 97, 0, 132, 5, 256, 61, 235, 87, 91, 193, 39, 111, 64, 180, 99, 95, 179, 35, 191, 246, 94, 238, 109, 22, 249, 187, 204, 245, 174, 53, 40, 86, 107, 119, 222, 239, 183, 157, 141, 123, 20, 196, 85, 83, 82, 167, 23, 139, 241, 72, 159, 2, 192, 175, 223, 54, 156, 73, 69, 208, 161, 120, 195, 158]

# Select the specified features using .iloc
X_selected = X.iloc[:, feature_indexes]

# Create a pipeline with the sampler and classifier
pipeline = ImbPipeline([
    ('sampler', RandomOverSampler()),  # Placeholder, will be set by GridSearchCV
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the extended parameter grid as a list of dictionaries
param_grid = [
    {
        # Include options with and without a sampler
        'sampler': [SMOTE(), ADASYN(), RandomOverSampler()],
        # Only include sampling_strategy if sampler is not None
        'sampler__sampling_strategy': ['auto', 0.5, 0.75, 1.0],

        # Classifier hyperparameters
        'classifier__n_estimators': [150, 200, 250, 300],  # Added 300 for more options
        'classifier__criterion': ['gini', 'entropy'],  # Added 'entropy' criterion for diversity
        'classifier__max_depth': [None] + list(range(5, 30, 10)),  # Depth extended to 45
        'classifier__min_samples_split': [2, 3, 5, 7],  # Added 2 as the default minimum split
        'classifier__min_samples_leaf': [1, 2, 3],  # Added 1 as the default minimum leaf size
        'classifier__max_features': ['log2', 'sqrt'],  # Added 'sqrt' for variety in feature selection
        'classifier__bootstrap': [False, True],  # Included True to test with bootstrapping
        'classifier__class_weight': [None, 'balanced']  # Added class weighting options
    },
    {
        'sampler': [None],

        # Classifier hyperparameters
        'classifier__n_estimators': [150, 200, 250, 300],  # Added 300 for more options
        'classifier__criterion': ['gini', 'entropy'],  # Added 'entropy' criterion for diversity
        'classifier__max_depth': [None] + list(range(5, 30, 10)),  # Depth extended to 45
        'classifier__min_samples_split': [2, 3, 5, 7],  # Added 2 as the default minimum split
        'classifier__min_samples_leaf': [1, 2, 3],  # Added 1 as the default minimum leaf size
        'classifier__max_features': ['log2', 'sqrt'],  # Added 'sqrt' for variety in feature selection
        'classifier__bootstrap': [False, True],  # Included True to test with bootstrapping
        'classifier__class_weight': [None, 'balanced']  # Added class weighting options
    }
]

# Define the scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'f1': 'f1'
}

# Define the Stratified 10-Fold Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calculate the number of parameter combinations
# This is the sum of the product of parameters in each dict in param_grid
n_param_combinations = 0
for grid in param_grid:
    n_combinations = 1
    for param in grid:
        n_combinations *= len(grid[param])
    n_param_combinations += n_combinations

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',  # Use 'roc_auc' to select the best model
    cv=cv,
    n_jobs=-1,  # Use all available cores
    verbose=0,  # Disable sklearn's verbose
    return_train_score=False
)

# Calculate the total number of parameter combinations for the progress bar
total_fits = int(
    sum(
        np.prod([len(values) for values in grid.values()])
        for grid in param_grid
    )
)

@contextmanager
def alive_joblib_bar(total):
    """
    Context manager to integrate alive-progress with joblib's Parallel processing.

    Parameters:
    - total: int, the total number of tasks to be processed.
    """
    with alive_bar(total, title='Grid Search Progress', bar='blocks', force_tty=True) as bar:
        # Store the original BatchCompletionCallBack.__call__ method
        original_callback = BatchCompletionCallBack.__call__
        lock = threading.Lock()

        def on_complete(self, *args, **kwargs):
            with lock:
                bar()
            return original_callback(self, *args, **kwargs)

        # Patch the BatchCompletionCallBack.__call__ method
        BatchCompletionCallBack.__call__ = on_complete
        try:
            yield
        finally:
            # Restore the original method to avoid side effects
            BatchCompletionCallBack.__call__ = original_callback

# Start the Grid Search with alive-progress
print("Starting Grid Search...")

with alive_joblib_bar(total_fits):
    grid_search.fit(X_selected, Y)

print("Grid Search Completed.")

# Retrieve the best average AUC and its standard deviation
best_auc = grid_search.best_score_
# Retrieve the standard deviation from cv_results_
# Identify the index of the best parameter set
best_index = grid_search.best_index_
best_auc_std = grid_search.cv_results_['std_test_roc_auc'][best_index]

# Retrieve the best hyperparameters
best_params = grid_search.best_params_

# Retrieve the other metrics for the best parameter set
best_accuracy = grid_search.cv_results_['mean_test_accuracy'][best_index]
best_accuracy_std = grid_search.cv_results_['std_test_accuracy'][best_index]

best_precision = grid_search.cv_results_['mean_test_precision'][best_index]
best_precision_std = grid_search.cv_results_['std_test_precision'][best_index]

best_f1 = grid_search.cv_results_['mean_test_f1'][best_index]
best_f1_std = grid_search.cv_results_['std_test_f1'][best_index]

# Output the results
print(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
print(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}")
print(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

Starting Grid Search...
Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▅▃ 373426/39936rid Search Progress |                                        | ▂▄▆ 0/39936 [0%]Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉                            | ▆▄▂ 11969/39936 Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▆█▆ 88449/39936 Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▃▁▃ 94977/39936 Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▄▂▂ 118241/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▅▃▁ 142401/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▇▅ 176449/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▂▂▄ 191457/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▄▂▂ 199233/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▇▅ 199521/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▂▄▆ 202977/39936Grid Se

/home/azureuser/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ (!) 399360/39936Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▃▁▃ 399360/39936
Grid Search Completed.

Best Average AUC: 0.7840 ± 0.0180
Average Accuracy: 0.8845 ± 0.0079
Average Precision: 0.3120 ± 0.0590
Average F1 Score: 0.2589 ± 0.0571

Best Hyperparameters:
  classifier__bootstrap: False
  classifier__class_weight: balanced
  classifier__criterion: entropy
  classifier__max_depth: None
  classifier__max_features: sqrt
  classifier__min_samples_leaf: 2
  classifier__min_samples_split: 7
  classifier__n_estimators: 200
  sampler: RandomOverSampler()
  sampler__sampling_strategy: 0.75


In [2]:
from sklearn.model_selection import cross_val_predict

# Retrieve the best estimator from Grid Search
best_estimator = grid_search.best_estimator_

# Use cross_val_predict to get cross-validated predicted probabilities
# Setting method='predict_proba' and using cv to ensure consistency
print("\nGenerating cross-validated predicted probabilities...")
y_pred_proba = cross_val_predict(best_estimator, X_selected, Y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

# Define a range of threshold values to evaluate
thresholds = np.linspace(0.0, 1.0, 101)

# Initialize variables to store the best metrics and threshold
best_threshold = 0.5
best_f1_score = 0.0
best_accuracy = 0.0
best_precision = 0.0

print("Optimizing threshold to maximize F1 score...")

for threshold in thresholds:
    # Convert predicted probabilities to binary predictions based on the threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate F1 score
    current_f1 = f1_score(Y, y_pred)
    
    # Update the best metrics and threshold if current F1 is better
    if current_f1 > best_f1_score:
        best_f1_score = current_f1
        best_threshold = threshold
        best_accuracy = accuracy_score(Y, y_pred)
        best_precision = precision_score(Y, y_pred, zero_division=0)

# Calculate standard deviations using cross-validation
# To compute standard deviations, we'll perform cross-validation predictions and calculate metrics at the best threshold

# Initialize lists to store per-fold metrics
f1_scores = []
accuracies = []
precisions = []

print("\nCalculating metrics at the optimal threshold across folds...")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_selected, Y), 1):
    # Split data
    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    
    # Fit the model on the training data
    best_estimator.fit(X_train, y_train)
    
    # Predict probabilities on the test data
    y_proba_fold = best_estimator.predict_proba(X_test)[:, 1]
    
    # Apply the optimal threshold
    y_pred_fold = (y_proba_fold >= best_threshold).astype(int)
    
    # Calculate metrics
    fold_f1 = f1_score(y_test, y_pred_fold)
    fold_accuracy = accuracy_score(y_test, y_pred_fold)
    fold_precision = precision_score(y_test, y_pred_fold, zero_division=0)
    
    # Append to lists
    f1_scores.append(fold_f1)
    accuracies.append(fold_accuracy)
    precisions.append(fold_precision)
    
    print(f"  Fold {fold}: F1={fold_f1:.4f}, Accuracy={fold_accuracy:.4f}, Precision={fold_precision:.4f}")

# Calculate mean and standard deviation for the metrics
mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

# Output the optimized threshold and corresponding metrics
print(f"\n=== Optimized Threshold ===")
print(f"Threshold for Maximum F1 Score: {best_threshold:.2f}")

print(f"\n=== Metrics at Optimal Threshold ===")
print(f"F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Precision: {mean_precision:.4f} ± {std_precision:.4f}")


Generating cross-validated predicted probabilities...
Optimizing threshold to maximize F1 score...

Calculating metrics at the optimal threshold across folds...
  Fold 1: F1=0.3690, Accuracy=0.8288, Precision=0.2793
  Fold 2: F1=0.3580, Accuracy=0.8317, Precision=0.2736
  Fold 3: F1=0.3444, Accuracy=0.8398, Precision=0.2737
  Fold 4: F1=0.3294, Accuracy=0.8155, Precision=0.2456
  Fold 5: F1=0.3537, Accuracy=0.8285, Precision=0.2685
  Fold 6: F1=0.2890, Accuracy=0.8010, Precision=0.2137
  Fold 7: F1=0.3567, Accuracy=0.8366, Precision=0.2772
  Fold 8: F1=0.3659, Accuracy=0.8317, Precision=0.2804
  Fold 9: F1=0.2981, Accuracy=0.8172, Precision=0.2308
  Fold 10: F1=0.3205, Accuracy=0.8285, Precision=0.2525

=== Optimized Threshold ===
Threshold for Maximum F1 Score: 0.17

=== Metrics at Optimal Threshold ===
F1 Score: 0.3385 ± 0.0267
Accuracy: 0.8259 ± 0.0110
Precision: 0.2595 ± 0.0219
